In [ ]:
# Install required packages
!pip install -q -U timm albumentations transformers opencv-python-headless


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 72.3 MB/s eta 0:00:00


import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
# show GPU (if available)
!nvidia-smi -L || true


In [ ]:
from google.colab import drive
drive.mount('/content/drive')   # follow the link and paste auth code

# Change DATA_DIR if your drive folder is different
DATA_DIR = '/content/drive/MyDrive/Flickr7kVersion'
import os
print("DATA_DIR exists:", os.path.exists(DATA_DIR))
print("Sample listing (first 20):", os.listdir(DATA_DIR)[:20])


Mounted at /content/drive
DATA_DIR exists: True
Sample listing (first 20): ['captions.txt', 'Images', 'weights_epoch_final.pt']


In [ ]:
import pandas as pd, os, random, cv2
captions_file = os.path.join(DATA_DIR, 'captions.txt')
assert os.path.exists(captions_file), f"captions.txt not found in {DATA_DIR}"

with open(captions_file, 'r', encoding='utf-8') as f:
    lines = [l.strip() for l in f if l.strip()]

get_captions = {}
all_captions = []

for line in lines:
    img_name = None
    caption = None
    # common formats:
    if '\t' in line:
        left, right = line.split('\t', 1)
        if '.jpg' in left:
            img_name = left.split('.jpg')[0].strip() + '.jpg'
        else:
            img_name = left.strip()
        caption = right.strip()
    elif '.jpg,' in line:
        left, right = line.split('.jpg,', 1)
        img_name = left.strip() + '.jpg'
        caption = right.strip()
    elif '.jpg#' in line:
        left, right = line.split('.jpg#', 1)
        img_name = left.strip() + '.jpg'
        if '\t' in right:
            _, caption = right.split('\t', 1)
        else:
            caption = right.strip()
    else:
        parts = line.split(' ', 1)
        if len(parts) == 2 and parts[0].endswith('.jpg'):
            img_name = parts[0]
            caption = parts[1].strip()
        else:
            # skip unexpected lines
            continue

    caption = caption.strip().strip('"')
    get_captions.setdefault(img_name, []).append(caption)
    all_captions.append(caption)

# DataFrame where each row = image filename + list of captions
df = pd.DataFrame(list(get_captions.items()), columns=['filename','caption'])
print("Total images parsed:", len(df))
print("Example row:", df.iloc[0].to_dict())
# optional: save a small CSV to inspect
df.head().to_csv('/content/df_head_sample.csv', index=False)
print("Saved sample df to /content/df_head_sample.csv (downloadable from left Files pane)")


Total images parsed: 8091
Example row: {'filename': '1000268201_693b08cb0e.jpg', 'caption': ['A child in a pink dress is climbing up a set of stairs in an entry way .', 'A girl going into a wooden building .', 'A little girl climbing into a wooden playhouse .', 'A little girl climbing the stairs to her playhouse .', 'A little girl in a pink dress going into a wooden cabin .']}
Saved sample df to /content/df_head_sample.csv (downloadable from left Files pane)


In [ ]:
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from transformers import AutoTokenizer
import cv2
import os, random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Tokenizer (no torchtext)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
context_length = 20

# Dataset
class ImageCaptioningDataset(Dataset):
    def __init__(self, df, data_dir, split='training', context_length=20):
        self.df = df.reset_index(drop=True)
        self.data_dir = data_dir
        self.context_length = context_length
        self.img_size = 224
        transforms = [A.Resize(self.img_size, self.img_size)]
        if split == 'training':
            transforms += [A.HorizontalFlip(), A.ColorJitter(), A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)), ToTensorV2()]
        else:
            transforms += [A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)), ToTensorV2()]
        self.transforms = A.Compose(transforms)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_filename, captions = self.df.iloc[idx]
        img_path = os.path.join(self.data_dir, 'Images', image_filename)
        img = cv2.imread(img_path)
        if img is None:
            raise FileNotFoundError(f"Image not found: {img_path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = self.transforms(image=img)['image']  # tensor HWC->CHW
        cap = random.choice(captions)
        enc = tokenizer(cap, max_length=self.context_length, padding='max_length', truncation=True, return_tensors='pt')
        input_ids = enc['input_ids'].squeeze(0)  # shape (context_length,)
        return img, input_ids

# Model
class ImageCaptioner(nn.Module):
    def __init__(self, context_length, vocabulary_size, num_blocks, model_dim, num_heads, prob):
        super().__init__()
        # EfficientNet feature extractor (no classifier)
        self.cnn_encoder = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0)
        # infer feature dim
        with torch.no_grad():
            dummy = torch.zeros(1,3,224,224)
            feat = self.cnn_encoder(dummy)
            in_features = feat.shape[1]

        self.project = nn.Linear(in_features, model_dim)
        self.word_embeddings = nn.Embedding(vocabulary_size, model_dim)
        self.pos_embeddings = nn.Embedding(context_length, model_dim)

        decoder_layer = nn.TransformerDecoderLayer(d_model=model_dim, nhead=num_heads,
                                                   dim_feedforward=2*model_dim, dropout=prob, batch_first=True)
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_blocks)
        self.vocab_projection = nn.Linear(model_dim, vocabulary_size)
        self.context_length = context_length

    def forward(self, images, true_labels):
        # images: (B,3,224,224), true_labels: (B,T)
        tok_embedded = self.word_embeddings(true_labels)           # (B,T,E)
        B, T = true_labels.shape
        positions = torch.arange(T, device=true_labels.device).unsqueeze(0).expand(B, T)
        pos_embedded = self.pos_embeddings(positions)
        total_embeddings = tok_embedded + pos_embedded            # (B,T,E)

        encoded_image = self.cnn_encoder(images)                  # (B, D)
        encoded_image = self.project(encoded_image)               # (B, E)
        img_for_attention = encoded_image.unsqueeze(1)            # (B, 1, E)

        tgt_mask = nn.Transformer.generate_square_subsequent_mask(T).to(images.device)
        out = self.decoder(total_embeddings, img_for_attention, tgt_mask=tgt_mask)  # (B,T,E)
        logits = self.vocab_projection(out)                       # (B,T,V)
        return logits

# Create dataset and dataloader (adjust batch_size if OOM)
train_dataset = ImageCaptioningDataset(df, DATA_DIR, split='training', context_length=context_length)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
print("Train dataset size:", len(train_dataset))


Device: cpu


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Train dataset size: 8091


In [ ]:
vocab_size = tokenizer.vocab_size
num_blocks = 6
model_dim = 512
num_heads = 8
prob = 0.1

model = ImageCaptioner(context_length, vocab_size, num_blocks, model_dim, num_heads, prob).to(device)

# Freeze CNN encoder to start (fine-tune later if you want)
for p in model.cnn_encoder.parameters():
    p.requires_grad = False

loss_fn = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
print("Model parameters (trainable):", sum(p.numel() for p in model.parameters() if p.requires_grad))


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Model parameters (trainable): 50877754


In [ ]:
model.train()
images, captions = next(iter(train_loader))
images = images.to(device)
captions = captions.to(device)            # (B, T)

inputs = captions[:, :-1]                 # decoder inputs
targets = captions[:, 1:]                 # next-token targets

preds = model(images, inputs)             # (B, T-1, V)
B, Tm1, V = preds.shape
loss = loss_fn(preds.view(B*Tm1, V), targets.reshape(-1))
print("Sanity loss:", loss.item())


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Sanity loss: 10.39645767211914


In [ ]:
num_epochs = 2
save_path = os.path.join(DATA_DIR, 'weights_epoch_final.pt')
num_iterations = 0

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for i, (images, captions) in enumerate(train_loader):
        images = images.to(device)
        captions = captions.to(device)
        optimizer.zero_grad()

        inputs = captions[:, :-1]
        targets = captions[:, 1:]
        preds = model(images, inputs)   # (B, T-1, V)

        B, Tm1, V = preds.shape
        loss = loss_fn(preds.view(B*Tm1, V), targets.reshape(-1))

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        optimizer.step()

        running_loss += loss.item()
        if num_iterations % 50 == 0:
            print(f"Epoch {epoch} Iter {num_iterations} loss: {loss.item():.4f}")
        num_iterations += 1

    epoch_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch} finished. Avg loss: {epoch_loss:.4f}")

# Save final weights to your Drive folder
torch.save(model.state_dict(), save_path)
print("Saved model to:", save_path)


In [ ]:
from google.colab import files
files.download(os.path.join(DATA_DIR, 'weights_epoch_final.pt'))


In [ ]:
import torch
from albumentations.pytorch import ToTensorV2

def generate_caption(model, image_path, tokenizer, max_len=20):
    model.eval()
    preprocess = A.Compose([A.Resize(224,224), A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)), ToTensorV2()])
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_t = preprocess(image=img)['image'].to(device)   # (3,224,224)
    with torch.no_grad():
        generated = [tokenizer.cls_token_id if tokenizer.cls_token_id is not None else tokenizer.bos_token_id]
        for _ in range(max_len):
            tgt = torch.tensor(generated, device=device).unsqueeze(0)  # (1, T)
            out = model(img_t.unsqueeze(0), tgt)  # (1, T, V)
            next_logits = out[0, -1, :]
            next_id = int(next_logits.argmax().item())
            generated.append(next_id)
            if next_id == tokenizer.sep_token_id or next_id == tokenizer.eos_token_id:
                break
        # decode and remove special tokens
        tokens_to_decode = [tok for tok in generated if tok not in (tokenizer.cls_token_id, tokenizer.sep_token_id, tokenizer.pad_token_id)]
        return tokenizer.decode(tokens_to_decode, skip_special_tokens=True)

# Example: test on one image
test_image = os.path.join(DATA_DIR, 'Images', df['filename'].iloc[7])
print("Test image:", test_image)
caption = generate_caption(model, test_image, tokenizer, max_len=context_length)
print("Generated caption:", caption)
# test_image.show();


Test image: /content/drive/MyDrive/Flickr7kVersion/Images/1012212859_01547e3f17.jpg
Generated caption: ##has # beetle beetle barak loki height pages barak grossed vigorously recurringnce episcopal azerbaijan recurringhips decrease encoding azerbaijan
